Faithfulness
Mengukur apakah jawaban model didukung oleh konteks.

Answer Relevancy
Mengukur apakah jawaban relevan dengan pertanyaan.

Context Recall
Mengukur apakah sistem retrieval berhasil mengambil informasi penting dari ground truth.

In [ ]:
# pip install ragas datasets langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
import os
from datasets import Dataset
from ragas import evaluate

# Tambahkan context_relevancy di sini
from ragas.metrics import faithfulness, answer_relevancy, context_recall

# Import library Langchain untuk custom endpoint (Maiarouter)
from langchain_openai.chat_models import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings

c:\Program Files\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\M S I\AppData\Local\Temp\ipykernel_15320\1540117385.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_recall
C:\Users\M S I\AppData\Local\Temp\ipykernel_15320\1540117385.py:7: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_recall
C:\Users\M S I\AppData\Local\

In [3]:
# 1. Sembunyikan warning sementara
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [4]:
# 2. Konfigurasi Endpoint Maiarouter (Pengaturan ini TIDAK diubah)
MAIAROUTER_API_KEY = "sk-I60768te05cyMuLkp30QMA"
MAIAROUTER_BASE_URL = "https://api.maiarouter.ai/v1"

# Inisialisasi LLM khusus untuk Ragas via Maiarouter
custom_llm = ChatOpenAI(
    api_key=MAIAROUTER_API_KEY,
    base_url=MAIAROUTER_BASE_URL,
    model="openai/gpt-3.5-turbo-16k",
    temperature=0.1
)

# Inisialisasi Embedding khusus untuk Ragas via Maiarouter
custom_embeddings = OpenAIEmbeddings(
    api_key=MAIAROUTER_API_KEY,
    base_url=MAIAROUTER_BASE_URL,
    model="openai/text-embedding-3-small"
)


In [5]:
# 3. Dataset murni (Skenario PerDesAI / Peraturan Desa)
data_samples = {
    "question": [
        "Bagaimana mekanisme pengangkatan perangkat desa menurut peraturan terbaru?",
        "Berapa lama masa jabatan kepala desa?"
    ],
    "contexts": [
        [
            "Perangkat desa diangkat oleh Kepala Desa setelah dikonsultasikan dengan Camat.",
            "Usia minimal 20 tahun dan maksimal 42 tahun."
        ],
        [
            "Kepala Desa menjabat selama 8 tahun sejak pelantikan.",
            "Desa memiliki PADes."
        ]
    ],
    "answer": [
        "Perangkat desa diangkat langsung oleh Kepala Desa tanpa perlu persetujuan pihak lain.",
        "Masa jabatan kepala desa adalah 8 tahun."
    ],
    "reference": [
        "Perangkat desa diangkat oleh Kepala Desa setelah konsultasi dengan Camat dan memenuhi syarat usia.",
        "Masa jabatan kepala desa adalah 8 tahun sejak pelantikan."
    ]
}

eval_dataset = Dataset.from_dict(data_samples)
print("Dataset berhasil disiapkan!")

# Masukkan ketiga metrik ke dalam list evaluasi
metrics = [
    faithfulness,
    answer_relevancy,
    context_recall
]

Dataset berhasil disiapkan!


In [6]:
# 4. Jalankan evaluasi dengan MENYUNTIKKAN custom_llm dan custom_embeddings
print("Menjalankan evaluasi Ragas (3 Metrik) via Maiarouter...")
try:
    evaluation_result = evaluate(
        dataset=eval_dataset,
        metrics=metrics,
        llm=custom_llm,
        embeddings=custom_embeddings
    )

    print("\n--- Hasil Evaluasi Keseluruhan ---")
    print(evaluation_result)

    df_results = evaluation_result.to_pandas()
    display(df_results.head())

except Exception as e:
    print(f"\nTerjadi error saat evaluasi: {e}")

Menjalankan evaluasi Ragas (3 Metrik) via Maiarouter...


Evaluating: 100%|██████████| 6/6 [00:36<00:00,  6.04s/it]



--- Hasil Evaluasi Keseluruhan ---
{'faithfulness': 0.5000, 'answer_relevancy': 0.6993, 'context_recall': 1.0000}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_recall
0,Bagaimana mekanisme pengangkatan perangkat des...,[Perangkat desa diangkat oleh Kepala Desa sete...,Perangkat desa diangkat langsung oleh Kepala D...,Perangkat desa diangkat oleh Kepala Desa setel...,0.0,0.398516,1.0
1,Berapa lama masa jabatan kepala desa?,[Kepala Desa menjabat selama 8 tahun sejak pel...,Masa jabatan kepala desa adalah 8 tahun.,Masa jabatan kepala desa adalah 8 tahun sejak ...,1.0,1.000000,1.0
